### conda env pedro_h2

Based on gemini conversation: How to answer the latest questions

# Analysis and Visualization Notebook - H₂ Electrocatalysis

**Objective:** This notebook uses the vector database enriched by scripts `1` to `4` to analyze, visualize, and answer research questions about the electrocatalysis literature.

In [ ]:
# --- 1. GLOBAL IMPORTS ---
import pandas as pd
import chromadb
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap
import os       # <-- Added to check file existence

import sys
sys.path.append(os.path.abspath("../02_scripts"))

import config

# --- 3. DATABASE CONNECTION ---
print(f"Connecting to collection: '{config.COLLECTION_NAME}'")
try:
    client_db = chromadb.PersistentClient(path=config.CHROMA_DB_PATH)
    collection = client_db.get_collection(name=config.COLLECTION_NAME)
    all_metadatas = collection.get(include=["metadatas"])['metadatas']
    df_analysis = pd.DataFrame(all_metadatas)
    
    # Ensures the 'doi_original' column is present for merging
    if 'doi_original' not in df_analysis.columns and 'doi' in df_analysis.columns:
        df_analysis['doi_original'] = df_analysis['doi'].str.split('_phrase_').str[0]
    
    print(f"ChromaDB data successfully loaded. Total of {len(df_analysis)} phrases.")

except Exception as e:
    print(f"ERROR loading ChromaDB data: {e}")
    df_analysis = pd.DataFrame()

# --- 2. LOADING, COUNT CALCULATION AND ADDITIONAL DATAFRAME JOIN ---
parquet_file = f'{config.DATA_FOLDER}/df_data.parquet' # Adjust the path if necessary

if not df_analysis.empty:
    try:
        print(f"\nLoading additional data from '{parquet_file}'...")
        df_data = pd.read_parquet(parquet_file)
        
        # --- CHANGE HERE: Renames the 'doi' column to 'doi_original' for standardization ---
        if 'doi' in df_data.columns and 'doi_original' not in df_data.columns:
            df_data.rename(columns={'doi': 'doi_original'}, inplace=True)
            print("Column 'doi' from Parquet file renamed to 'doi_original' for the join.")

        # --- The rest of the logic will now work correctly ---

        print("Calculating phrase count by topic...")
        topic_counts = df_data['topic_name_cluster3'].value_counts().reset_index()
        topic_counts.columns = ['topic_name_cluster3', 'topic_count']
        
        df_topic_info = df_data[['topic_name_cluster3', 'topic_name_cluster3_reduced']].drop_duplicates()
        df_topic_info = pd.merge(df_topic_info, topic_counts, on='topic_name_cluster3', how='left')

        df_join_data = df_data[['doi_original', 'year', 'topic_name_cluster3', 'first_country']].drop_duplicates(subset=['doi_original'])
        df_join_data = pd.merge(df_join_data, df_topic_info, on='topic_name_cluster3', how='left')

        print("Joining data...")
        df_analysis = pd.merge(
            df_analysis,
            df_join_data.drop(columns=['topic_name_cluster3']),
            on='doi_original',
            how='left'
        )
        
        print("Join successfully completed.")
        
        display(df_analysis[['doi', 'year', 'topic_count', 'topic_name_cluster3_reduced']].head(3))

    except FileNotFoundError:
        print(f"WARNING: The file '{parquet_file}' was not found.")
    except Exception as e:
        print(f"ERROR processing the Parquet file: {e}")

In [ ]:
# Display dataframe columns
df_analysis.columns

---
## Question: What are the main technological and economic challenges?

To answer this question, we will follow a three-step approach:
1.  **Semantic Grouping:** We will use embeddings to group challenge labels (`challenge_label`) that are semantically similar (e.g., "low stability" and "poor durability"). This consolidates the "long tail" of challenges and gives us a clearer view of the main categories.
2.  **Ranking (Table):** We will create a table with the ranking of the most cited challenges, with the labels already grouped.
3.  **Visualization (Chart):** We will generate a bar chart to visually illustrate the frequency of each challenge category.

In [ ]:
# --- Font size definition ---
ft1 = 48 # Font for main title
ft2 = 42 # Font for axis titles (X and Y)
ft3 = 38 # Font for axis labels/ticks (X and Y)

# --- Name of the column with challenges already grouped by script 6 ---
grouped_challenge_col = f'ext_{config.LLM_ADVANCED_MODEL_SAFE_NAME}_challenge_group'

if grouped_challenge_col not in df_analysis.columns:
    print(f"ERROR: Column '{grouped_challenge_col}' not found.")
    print("Make sure script '6_group_challenges.py' was executed and Cell 2 of this notebook was updated to load the latest data.")
else:
    print(f"\n--- Table: Top 20 Most Cited Challenge Categories (from '{grouped_challenge_col}') ---")
    
    # Filters null or 'Uncategorized' data in grouped column
    df_ranked = df_analysis.dropna(subset=[grouped_challenge_col])
    df_ranked = df_ranked[df_ranked[grouped_challenge_col] != 'Uncategorized']
    
    # Creates ranking using value_counts() on grouped column
    challenge_ranking = df_ranked[grouped_challenge_col].value_counts().head(20)
    
    # Converts to DataFrame for better display
    df_ranking_table = challenge_ranking.to_frame(name="Number of Mentions")
    
    total_top20 = df_ranking_table["Number of Mentions"].sum()
    df_ranking_table["Percentage (%)"] = (
        df_ranking_table["Number of Mentions"] / total_top20 * 100
    ).round(2)

    display(df_ranking_table)
    
    # --- Preparation and export to LaTeX with new names ---
    df_latex = df_ranking_table.copy()
    df_latex.index.name = "Challenge"
    df_latex.rename(columns={"Number of Mentions": "Frequency"}, inplace=True)
    
    # Ensures directory exists before saving
    df_latex.to_latex(f'{config.RESULTS_FOLDER}/q4_14_1_challenge_ranking_table.tex',
                      float_format="%.2f")

    # --- Chart: Ranking Visualization ---
    print("\n--- Chart: Top 20 Most Cited Challenge Categories (from Manual Grouping) ---")
    
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(10, 14))
    
    ax = sns.barplot(
        x=challenge_ranking.values,
        y=challenge_ranking.index,
        palette="viridis"
    )
    
    # Defines a small offset for text
    offset = ax.get_xlim()[1] * 0.01

    for p in ax.patches:
        width = p.get_width()
        
        # If bar is too long, puts text inside
        if width > 34000:
            ax.text(width - offset, # X position (slightly before end of bar)
                    p.get_y() + p.get_height() / 2., # Y position (middle of bar)
                    f'{int(width)}',
                    ha='right', # Right align
                    va='center',
                    fontsize=ft3,
                    color='white') # White color for contrast
        # If short, puts text outside
        else:
            ax.text(width + offset, # X position (end of bar + offset)
                    p.get_y() + p.get_height() / 2.,
                    f'{int(width)}',
                    ha='left', # Left align
                    va='center',
                    fontsize=ft3,
                    color='black') # Black color

    # Gets current Y axis labels
    y_labels = [label.get_text() for label in ax.get_yticklabels()]
    # Applies line wrap to each label
    wrapped_labels = [textwrap.fill(label, width=50) for label in y_labels]
    # Sets new labels with line break on chart
    ax.set_yticklabels(wrapped_labels)

    plt.title('Ranking of Main Technological\nand Economic Challenges', fontsize=ft1, pad=20)
    plt.xlabel('Number of Sentences', fontsize=ft2)
    plt.ylabel('Challenge Category', fontsize=ft2)
    
    plt.xticks(fontsize=ft3, rotation=45)
    plt.yticks(fontsize=ft3)

    # Adjusts X axis limit
    #plt.xlim(right=ax.get_xlim()[1] * 1.15)
    
    # tight_layout is important after line wrap to adjust space
    plt.tight_layout()
        
    plt.savefig(f'{config.RESULTS_FOLDER}/q4_14_1_challenge_ranking_chart.pdf', dpi=300, bbox_inches='tight')
    plt.show()

### Additional Analysis: Challenge Heatmap by Research Topic

This visualization gives us an overview of the entire database, showing the intersection between **Research Topics** (rows) and **Challenge Categories** (columns). The color intensity indicates the frequency with which a specific challenge is mentioned within a given topic, allowing us to quickly identify the main problem areas for each field of study.

In [ ]:
##########################################################
#### heatmap by topics
##########################################################

import textwrap

def wrap_labels(labels, width=20):
    return ['\n'.join(textwrap.wrap(str(label), width)) for label in labels]

# --- Font size definition ---
ft1 = 50 # Font for main title
ft2 = 40 # Font for axis titles (X and Y)
ft3 = 38 # Font for axis labels/ticks AND ANNOTATIONS
ft4 = 22
limit_to_show = 1

# Column keys
challenge_col = f'ext_{config.LLM_ADVANCED_MODEL_SAFE_NAME}_challenge_group'
topic_col = 'topic_name_cluster3_reduced'

if challenge_col not in df_analysis.columns or topic_col not in df_analysis.columns:
    print("ERROR: Required columns not found.")
else:
    # --- 1. DATA PREPARATION AND SORTING ---
    df_filtered = df_analysis.dropna(subset=[challenge_col, topic_col])
    
    TOP_N_TOPICS = 100
    TOP_N_CHALLENGES = 15

    top_topics_sorted = df_filtered.groupby(topic_col)['topic_count'].first().sort_values(ascending=False).head(TOP_N_TOPICS).index
    top_challenges_sorted = df_filtered[challenge_col].value_counts().head(TOP_N_CHALLENGES).index
    
    df_heatmap = df_filtered[
        (df_filtered[topic_col].isin(top_topics_sorted)) &
        (df_filtered[challenge_col].isin(top_challenges_sorted))
    ]

    if df_heatmap.empty:
        print("No data found for selected filters.")
    else:
        # --- 2. CONTINGENCY TABLE CREATION AND REORDERING ---
        topic_challenge_crosstab = pd.crosstab(
            df_heatmap[topic_col],
            df_heatmap[challenge_col]
        )
        
        topic_challenge_crosstab = topic_challenge_crosstab.reindex(
            index=top_topics_sorted.intersection(topic_challenge_crosstab.index),
            columns=top_challenges_sorted.intersection(topic_challenge_crosstab.columns)
        )

        print(f"\n--- Table: Count of Top {TOP_N_CHALLENGES} Challenges in Top {TOP_N_TOPICS} Topics (sorted) ---")
        display(topic_challenge_crosstab)

        # --- EXPORT TO LATEX ---
        try:
            topic_challenge_crosstab.to_latex(f'{config.RESULTS_FOLDER}/q4_14_1_topic_vs_challenges.tex') #, longtable=True)
            print("\nTable exported to 'topic_vs_challenges.tex'")
        except Exception as e:
            print(f"\nError exporting to LaTeX: {e}")


        # --- 3. HEATMAP GENERATION ---
        print(f"\n--- Chart: Heatmap of Challenges by Topic ---")
        
        # --- CHANGE HERE: Creates conditional annotations matrix ---
        # Creates copy of table, but with formatted strings
        # Leaves cell blank if value <= 1000
        annot_labels = topic_challenge_crosstab.applymap(lambda val: f'{val:d}' if val > limit_to_show else '')
        
        plt.figure(figsize=(24, 26))
        
        # --- CHANGE HERE: Uses new parameters in heatmap ---
        ax = sns.heatmap(
            topic_challenge_crosstab,
            #annot=annot_labels, # Uses custom annotations matrix
            fmt='s',            # Uses string format since matrix contains strings
            cmap='rocket_r',
            annot_kws={'size': ft4} # Sets annotation font size
        )
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=ft3)        

        plt.subplots_adjust(left=0.5, bottom=0.40)

        ax.set_xticklabels(wrap_labels(topic_challenge_crosstab.columns, width=50))
        ax.set_yticklabels(wrap_labels(topic_challenge_crosstab.index, width=60))

        plt.title('Heatmap of Challenges across\nTop Research Topics', fontsize=ft1, pad=40)
        plt.xlabel('Challenge Category', fontsize=ft2)
        plt.ylabel('Research Topic', fontsize=ft2)
        
        plt.xticks(rotation=45, ha='right', fontsize=ft3)
        plt.yticks(fontsize=ft3)
        
        plt.savefig(f'{config.RESULTS_FOLDER}/q4_14_1_topic_vs_challenges_heatmap.pdf', dpi=300, bbox_inches='tight')
        plt.show()


In [ ]:
##########################################################
#### heatmap by country
##########################################################

# --- Font size definition ---
ft1 = 44 # Font for main title
ft2 = 38 # Font for axis titles (X and Y)
ft3 = 36 # Font for axis labels/ticks AND ANNOTATIONS
ft4 = 22
limit_to_show = 1

# Column keys
challenge_col = f'ext_{config.LLM_ADVANCED_MODEL_SAFE_NAME}_challenge_group'
country_col = 'first_country' # Renamed for clarity

if challenge_col not in df_analysis.columns or country_col not in df_analysis.columns:
    print("ERROR: Required columns not found.")
else:
    # --- 1. DATA PREPARATION AND SORTING ---
    df_filtered = df_analysis.dropna(subset=[challenge_col, country_col])
    df_filtered = df_filtered[df_filtered["first_country"] != "UNKNOWN"]

    df_filtered = df_filtered[df_filtered[challenge_col] != "Uncategorized"]

    # --- CHANGE HERE: Defines number of countries to show ---
    TOP_N_COUNTRIES = 20
    TOP_N_CHALLENGES = 15

    # --- CHANGE HERE: Finds top N countries by occurrence count ---
    top_countries_sorted = df_filtered[country_col].value_counts().head(TOP_N_COUNTRIES).index
    # Finds top N challenges
    top_challenges_sorted = df_filtered[challenge_col].value_counts().head(TOP_N_CHALLENGES).index

    # Filters DataFrame to include only selected countries and challenges
    df_heatmap = df_filtered[
        (df_filtered[country_col].isin(top_countries_sorted)) &
        (df_filtered[challenge_col].isin(top_challenges_sorted))
    ]

    if df_heatmap.empty:
        print("No data found for selected filters.")
    else:
        # --- 2. CONTINGENCY TABLE CREATION AND REORDERING ---
        country_challenge_crosstab = pd.crosstab(
            df_heatmap[country_col],
            df_heatmap[challenge_col]
        )

        # Reorders rows (countries) and columns (challenges) by frequency
        country_challenge_crosstab = country_challenge_crosstab.reindex(
            index=top_countries_sorted.intersection(country_challenge_crosstab.index),
            columns=top_challenges_sorted.intersection(country_challenge_crosstab.columns)
        )

        print(f"\n--- Table: Count of Top {TOP_N_CHALLENGES} Challenges in Top {TOP_N_COUNTRIES} Countries (sorted) ---")
        display(country_challenge_crosstab)

        # --- EXPORT TO LATEX ---
        try:
            country_challenge_crosstab.to_latex(f'{config.RESULTS_FOLDER}/q4_14_1_first_country_vs_challenges.tex')
            print("\nTable exported to 'first_country_vs_challenges.tex'")
        except Exception as e:
            print(f"\nError exporting to LaTeX: {e}")

        # --- 3. HEATMAP GENERATION ---
        print(f"\n--- Chart: Heatmap of Challenges by Country ---")

        annot_labels = country_challenge_crosstab.applymap(lambda val: f'{val:d}' if val > limit_to_show else '')

        plt.figure(figsize=(18, 18))

        ax = sns.heatmap(
            country_challenge_crosstab,
            #annot=annot_labels,
            fmt='s',
            cmap='rocket_r',
            #annot_kws={'size': ft4}
        )
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=ft3)

        plt.title('Heatmap of Challenges across Top 20 Countries', fontsize=ft1, pad=20)
        plt.xlabel('Challenge Category', fontsize=ft2)
        plt.ylabel('Country', fontsize=ft2) # Updated Y axis label

        plt.xticks(rotation=45, ha='right', fontsize=ft3)
        plt.yticks(fontsize=ft3)

        plt.savefig(f'{config.RESULTS_FOLDER}/q4_14_1_first_country_vs_challenges_heatmap.pdf', dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
# (Assuming df_analysis, config, pd, sns, plt are already defined)

# --- Font size definition ---
ft1 = 54  # Font for main title
ft2 = 40  # Font for axis titles (X and Y)
ft3 = 36  # Font for axis labels/ticks and annotations
ft4 = 36
limit_to_show = 1  # Minimum value to show in annotation (for 'absolute')

# ########################################################
# ###           HEATMAP CONFIGURATION                ###
# ########################################################
#
# Choose metric for COLOR SCALE
# Options: 'absolute', 'percent_total', 'percent_row', 'percent_col', 'z_score_total', 'z_score_row', 'z_score_col' # <--- ADDITION
COLOR_SCALE_METRIC = 'percent_row' # <--- Example of new metric usage
#
# Choose metric for TEXT (ANNOTATION)
# Options: 'absolute', 'percent_total', 'percent_row', 'percent_col', 'z_score_total', 'z_score_row', 'z_score_col' # <--- ADDITION
ANNOTATION_METRIC = 'percent_row' # <--- Example of new metric usage
#
# ########################################################

# Column keys
challenge_col = f'ext_{config.LLM_ADVANCED_MODEL_SAFE_NAME}_challenge_group'
country_col = 'first_country'

if challenge_col not in df_analysis.columns or country_col not in df_analysis.columns:
    print("ERROR: Required columns not found.")
else:
    # --- 1. DATA PREPARATION AND SORTING ---
    df_filtered = df_analysis.dropna(subset=[challenge_col, country_col])
    df_filtered = df_filtered[df_filtered[country_col] != "UNKNOWN"]

    df_filtered = df_filtered[df_filtered[challenge_col] != "Uncategorized"]
    
    # Top N countries and challenges
    TOP_N_COUNTRIES = 20
    TOP_N_CHALLENGES = 15

    top_countries_sorted = df_filtered[country_col].value_counts().head(TOP_N_COUNTRIES).index
    top_challenges_sorted = df_filtered[challenge_col].value_counts().head(TOP_N_CHALLENGES).index

    # Filters only top countries and challenges
    df_heatmap = df_filtered[
        (df_filtered[country_col].isin(top_countries_sorted)) &
        (df_filtered[challenge_col].isin(top_challenges_sorted))
    ]

    if df_heatmap.empty:
        print("No data found for selected filters.")
    else:
        # --- 2. CONTINGENCY TABLE CREATION (Absolute Values) ---
        country_challenge_crosstab = pd.crosstab(
            df_heatmap[country_col],
            df_heatmap[challenge_col]
        )

        # Reorders rows and columns
        country_challenge_crosstab = country_challenge_crosstab.reindex(
            index=top_countries_sorted.intersection(country_challenge_crosstab.index),
            columns=top_challenges_sorted.intersection(country_challenge_crosstab.columns)
        )

        print(f"\n--- Table: Count of Top {TOP_N_CHALLENGES} Challenges in Top {TOP_N_COUNTRIES} Countries (sorted) ---")
        display(country_challenge_crosstab)

        # --- EXPORT TO LATEX ---
        try:
            country_challenge_crosstab.to_latex(f'{config.RESULTS_FOLDER}/q4_14_1_first_country_vs_challenges.tex')
            print("\nTable exported to 'first_country_vs_challenges.tex'")
        except Exception as e:
            print(f"\nError exporting to LaTeX: {e}")

        # --- 3. CALCULATION OF ALL METRICS ---
        
        # 1. 'absolute': Absolute Values
        metric_absolute = country_challenge_crosstab.copy()
        
        # 2. 'percent_total': Percentage of Total
        total_sum = metric_absolute.sum().sum()
        metric_percent_total = (metric_absolute / total_sum) * 100
        
        # 3. 'z_score_total': Z-Score based on mean and std of ENTIRE table
        all_values = metric_absolute.values.flatten()
        total_mean = all_values.mean()
        total_std = all_values.std(ddof=0)
        metric_z_score_total = (metric_absolute - total_mean) / total_std if total_std > 0 else (metric_absolute * 0)

        # 4. 'z_score_row': Z-Score by Row (Country) - (Your "pearson by row")
        metric_z_score_row = metric_absolute.apply(lambda row: (row - row.mean()) / row.std(ddof=0), axis=1).fillna(0)
        
        # 5. 'z_score_col': Z-Score by Column (Challenge) - (Your "pearson by column" / original r_matrix)
        metric_z_score_col = metric_absolute.apply(lambda col: (col - col.mean()) / col.std(ddof=0), axis=0).fillna(0)
        
        # 6. 'percent_row': Percentage by Row (Country) # <--- ADDITION
        metric_percent_row = metric_absolute.div(metric_absolute.sum(axis=1), axis=0).mul(100).fillna(0)
        
        # 7. 'percent_col': Percentage by Column (Challenge) # <--- ADDITION
        metric_percent_col = metric_absolute.div(metric_absolute.sum(axis=0), axis=1).mul(100).fillna(0)

        
        # Dictionary for easy access
        metrics_data = {
            'absolute': metric_absolute,
            'percent_total': metric_percent_total,
            'z_score_total': metric_z_score_total,
            'z_score_row': metric_z_score_row,
            'z_score_col': metric_z_score_col,
            'percent_row': metric_percent_row, # <--- ADDITION
            'percent_col': metric_percent_col  # <--- ADDITION
        }
        
        # Validation of choices
        if COLOR_SCALE_METRIC not in metrics_data:
            print(f"ERROR: Invalid COLOR_SCALE_METRIC '{COLOR_SCALE_METRIC}'. Using 'z_score_col'.")
            COLOR_SCALE_METRIC = 'z_score_col'
        if ANNOTATION_METRIC not in metrics_data:
            print(f"ERROR: Invalid ANNOTATION_METRIC '{ANNOTATION_METRIC}'. Using 'absolute'.")
            ANNOTATION_METRIC = 'absolute'
        
        # Selects dataframes based on configs
        df_color = metrics_data[COLOR_SCALE_METRIC]
        df_annot_data = metrics_data[ANNOTATION_METRIC]

        # --- 4. HEATMAP PREPARATION AND PLOTTING ---
        print(f"\n--- Chart: Heatmap (Color: {COLOR_SCALE_METRIC}, Annot: {ANNOTATION_METRIC}) ---")

        # --- A. Prepare Annotations (Text) ---
        fmt_str = 's' # We use 's' (string) since labels are pre-formatted
        
        if ANNOTATION_METRIC == 'absolute':
            annot_labels = df_annot_data.applymap(lambda val: f'{val:d}' if val >= limit_to_show else '')
        
        # <--- CHANGE: Grouping all percentage types ---
        elif ANNOTATION_METRIC in ['percent_total', 'percent_row', 'percent_col']:
            # Limit of 0.081 kept from original code for 'percent_total'.
            # You might want to adjust this or use a different limit for row/col.
            limit_percent = 0.081 
            annot_labels = df_annot_data.applymap(lambda val: f'{val:.0f}' if val >= limit_percent else '')
            
        elif ANNOTATION_METRIC in ['z_score_total', 'z_score_row', 'z_score_col']:
            # Ex: show only z-scores with |z| > 0.05
            annot_labels = df_annot_data.applymap(lambda val: f'{val:.2f}' if abs(val) >= 0.05 else '') 
        else:
            annot_labels = df_annot_data.applymap(str) # Fallback

        # --- B. Prepare Color Scale Parameters ---
        cbar_label = f'Metric: {COLOR_SCALE_METRIC}'
        cbar_kws = {'label': cbar_label}
        center = None
        
        if COLOR_SCALE_METRIC == 'z_score_col': 
            # Original script case (Z-Score by column, treated as 'r')
            cmap = 'coolwarm'
            vmin = -1 # Fixed limit, as in your original
            vmax = 1
            center = 0.0
            cbar_kws = {'label': 'Z-Score (Column-wise, clamped -1 to 1)'}
            
        elif COLOR_SCALE_METRIC in ['z_score_row', 'z_score_total']:
            # Z-scores (row or total) with symmetric scale based on data
            cmap = 'coolwarm'
            # Finds largest absolute value to center at 0
            abs_max = max(abs(df_color.min().min()), abs(df_color.max().max()))
            vmin = -abs_max
            vmax = abs_max
            center = 0.0
            cbar_kws = {'label': f'Z-Score ({COLOR_SCALE_METRIC})'}
            
        elif COLOR_SCALE_METRIC == 'percent_total':
            cmap = 'YlGnBu' # Sequential
            vmin = 0
            vmax = df_color.max().max() # Max percentage (rarely 100)
            cbar_kws = {'label': 'Percentage (of Total)'}
        
        # <--- ADDITION: Logic for new percentage scales ---
        elif COLOR_SCALE_METRIC in ['percent_row', 'percent_col']:
            cmap = 'YlGnBu' # Sequential
            vmin = 0
            vmax = 100 # Row/col percentage scale always goes from 0 to 100
            label = "" #'Percentage (by Row)' if COLOR_SCALE_METRIC == 'percent_row' else 'Percentage (by Column)'
            cbar_kws = {'label': label}
            
        else: 
            # 'absolute' (Default)
            cmap = 'YlGnBu' # Sequential
            vmin = 0
            vmax = df_color.max().max()
            cbar_kws = {'label': 'Absolute Count'}


        # --- C. Plot Heatmap ---
        plt.figure(figsize=(20, 20))
        ax = sns.heatmap(
            df_color,                      # DATA FOR COLOR (based on COLOR_SCALE_METRIC)
            annot=annot_labels,            # DATA FOR TEXT (based on ANNOTATION_METRIC)
            fmt=fmt_str,                   # 's' since it's pre-formatted
            cmap=cmap,
            vmin=vmin, 
            vmax=vmax,
            center=center,
            annot_kws={'size': ft4},
            cbar_kws=cbar_kws
        )

        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=ft4)

        plt.title("Percentage Share by Country", fontsize=ft1, pad=20) #(f'Heatmap (Color: {COLOR_SCALE_METRIC}, Annot: {ANNOTATION_METRIC})', fontsize=ft1, pad=20)
        plt.xlabel('Challenge Category', fontsize=ft2)
        plt.ylabel('Country', fontsize=ft2)
        plt.xticks(rotation=45, ha='right', fontsize=ft3)
        plt.yticks(fontsize=ft3)

        # Dynamic filename
        output_filename = f'{config.RESULTS_FOLDER}/q4_14_1_heatmap_color_{COLOR_SCALE_METRIC}_annot_{ANNOTATION_METRIC}.pdf'
        
        plt.savefig(output_filename, dpi=300, bbox_inches='tight')
        print(f"\nHeatmap saved to: {output_filename}")
        plt.show()


In [ ]:
#######################################
# Hierarchical clustering study of challenges
#######################################

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
import numpy as np


fs1 = 70
fs2 = 55
fs3 = 50
fs4 = 55

# --- 1. DATA PREPARATION AND SORTING (Copied from original script) ---
df_filtered = df_analysis.dropna(subset=[challenge_col, country_col])
df_filtered = df_filtered[df_filtered[country_col] != "UNKNOWN"]

top_countries_sorted = df_filtered[country_col].value_counts().head(TOP_N_COUNTRIES).index
top_challenges_sorted = df_filtered[challenge_col].value_counts().head(TOP_N_CHALLENGES).index

df_heatmap = df_filtered[
    (df_filtered[country_col].isin(top_countries_sorted)) &
    (df_filtered[challenge_col].isin(top_challenges_sorted))
]

# --- 2. CONTINGENCY TABLE CREATION (Copied from original script) ---
country_challenge_crosstab = pd.crosstab(
    df_heatmap[country_col],
    df_heatmap[challenge_col]
)
# Reorders rows and columns for consistency
country_challenge_crosstab = country_challenge_crosstab.reindex(
    index=top_countries_sorted.intersection(country_challenge_crosstab.index),
    columns=top_challenges_sorted.intersection(country_challenge_crosstab.columns)
)

print("\n--- Count Table (Original) ---")
print(country_challenge_crosstab.head())

uncategorized_col_name = 'Uncategorized' # Check if name is exact
if uncategorized_col_name in country_challenge_crosstab.columns:
    country_challenge_crosstab = country_challenge_crosstab.drop(columns=[uncategorized_col_name])
    print(f"\n--- Column '{uncategorized_col_name}' removed from analysis ---")

##################################################################
#### START OF NEW CODE - HIERARCHICAL CLUSTERING
##################################################################

print("\n--- Starting Hierarchical Clustering ---")

# --- STEP 1: PROFILE NORMALIZATION (BY ROW) ---
# This removes the volume effect (hegemony) and focuses on the "research profile".
# We divide each challenge count of a country by the total challenges of that country.

# Calculates row total (country)
row_totals = country_challenge_crosstab.sum(axis=1)

# Filters countries that might have 0 challenges (avoids division by zero)
countries_with_data = row_totals[row_totals > 0].index
country_challenge_crosstab_filtered = country_challenge_crosstab.loc[countries_with_data]
row_totals_filtered = row_totals.loc[countries_with_data]

# Divides cell count by row total to get proportion
df_profile = country_challenge_crosstab_filtered.div(row_totals_filtered, axis=0)

# Fills any NaN that might arise with 0
df_profile = df_profile.fillna(0)

print("\n--- Profile Table (Row Normalized) ---")
print(df_profile.head())

# --- STEP 2: GENERATE CLUSTERMAP (Recommended Solution) ---
# This creates a heatmap of *profiles* and adds dendrograms
# grouping countries (rows) and challenges (columns) by similarity.

print("\n--- Generating Clustermap (Heatmap + Dendrogram) ---")

# Adjust figsize to fit your number of labels
# method='ward' is a robust clustering technique
# metric='euclidean' is the standard distance
# cmap='YlGnBu' is a good color palette for proportions (0 to 1)

#cbar_position_horizontal = (0.25, 0.08, 0.5, 0.03)
cbar_position_horizontal = (0.70, 0.95, 0.5, 0.03)

g = sns.clustermap(
    df_profile,
    method='ward',
    metric='euclidean',
    cmap='YlGnBu',  # Boa paleta para perfis (baixo para alto)
    figsize=(18, 24),
    annot=False,  # Annotations would be too cluttered in a dense plot
    #cbar_kws={'label': 'Proportion of the Country’s Research Effort'}
    cbar_pos=cbar_position_horizontal,
    cbar_kws={'orientation': 'horizontal'}
)

for dendro_ax in [g.ax_row_dendrogram, g.ax_col_dendrogram]:
    for line in dendro_ax.collections:
        line.set_linewidth(4)

g.cax.set_xlabel('Proportion of the Country’s\nResearch Effort', fontsize=fs2)
g.cax.tick_params(labelsize=fs3)

# Add titles and adjust labels
g.figure.suptitle('Hierarchical Clustering of Countries by Challenge Profile', y=1.05, fontsize=fs1)
g.ax_heatmap.set_xlabel('Challenge Category', fontsize=fs2)
g.ax_heatmap.set_ylabel('Country', fontsize=fs2)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=fs4)
plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=fs4)

# Save figure
cluster_filename = f'{config.RESULTS_FOLDER}/q4_14_1_hierarchical_clustermap_profiles.pdf'
plt.savefig(cluster_filename, dpi=300, bbox_inches='tight')
print(f"Clustermap saved to: {cluster_filename}")
plt.show()

# --- STEP 3: (Alternative) Plot Only Dendrogram ---
# In case you *only* want the clustering tree, without the heatmap.

print("\n--- Generating only Dendrogram (Alternative) ---")

# Calculate linkage matrix
Z = linkage(df_profile, method='ward', metric='euclidean')

plt.figure(figsize=(12, 30))
dendrogram(
    Z,
    labels=df_profile.index,
    orientation='right',  # 'right' or 'left' is better for many countries
    leaf_font_size=14,
)

plt.title('Similarity Dendrogram of Research Profile by Country', fontsize=18, pad=20)
plt.xlabel('Euclidean Distance ("Ward" Clustering)', fontsize=14)
plt.ylabel('Country', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Save figure
dendro_filename = f'{config.RESULTS_FOLDER}/q4_14_1_hierarchical_dendrogram_only.pdf'
plt.savefig(dendro_filename, dpi=300, bbox_inches='tight')
print(f"Dendrogram saved to: {dendro_filename}")
plt.show()

### Temporal Analysis: Evolution of Challenges Over the Years

This analysis investigates how the frequency of main research challenges changed over time. A table will show the citation count of each challenge by year, and a line chart will visualize the trends of the most prominent challenges.

In [ ]:
##########################################################
#### Cell 13
##########################################################


# --- Font size definition ---
ft1 = 48 # Font for main title
ft2 = 40 # Font for axis titles (X and Y)
ft3 = 36 # Font for axis labels/ticks (X and Y)

# --- CHANGE HERE: Variable to configure chart type ---
# Choose 'absolute' (absolute values), 'relative' (percentage per year) or 'base100' (base 100 index).
PLOT_TYPE = 'relative' #'absolute'  # or 'relative' or 'base100'

# --- CHANGE HERE: Variable to define how many lines to plot ---
NUM_CHALLENGES_TO_PLOT = 15

# Column keys
challenge_col = f'ext_{config.LLM_ADVANCED_MODEL_SAFE_NAME}_challenge_group'
year_col = 'year'
doi_original_col = 'doi_original'

if challenge_col not in df_analysis.columns or year_col not in df_analysis.columns:
    print("ERROR: 'challenge_grouped' or 'year' columns not found. Run previous cells.")
else:
    # --- 1. DATA PREPARATION ---
    df_temporal = df_analysis.dropna(subset=[challenge_col, year_col])
    df_temporal[year_col] = df_temporal[year_col].astype(int)
    df_temporal = df_temporal[df_temporal[year_col] >= 2014]

    top_15_challenges = challenge_ranking.head(15).index
    df_temporal_top = df_temporal[df_temporal[challenge_col].isin(top_15_challenges)]

    # --- 2. ABSOLUTE CROSSTAB ---
    temporal_crosstab = pd.crosstab(
        df_temporal_top[year_col],
        df_temporal_top[challenge_col]
    )
    
    sorted_columns = [col for col in top_15_challenges if col in temporal_crosstab.columns]
    temporal_crosstab = temporal_crosstab[sorted_columns]

    print("\n--- Table: Absolute Frequency of Top 15 Challenges per Year ---")
    display(temporal_crosstab)
    
    # --- 3. DEFINITION OF PLOTTING DATA ---
    if PLOT_TYPE == 'relative':
        print(f"\n--- Chart: RELATIVE Evolution (per Publication) of Top {NUM_CHALLENGES_TO_PLOT} Challenges ---")
        
        total_pubs_per_year = df_temporal.groupby(year_col)[doi_original_col].nunique()
        challenge_pubs_per_year = df_temporal_top.groupby([year_col, challenge_col])[doi_original_col].nunique()
        numerator_crosstab = challenge_pubs_per_year.unstack(fill_value=0)
        plot_crosstab = numerator_crosstab.div(total_pubs_per_year, axis=0).fillna(0) * 100
        
        y_axis_label = '% of Publications'
        plot_title = f'Penetration of Top {NUM_CHALLENGES_TO_PLOT} Challenges in Literature'
        plot_crosstab.to_latex(f'{config.RESULTS_FOLDER}/q4_14_1_challenges_relative_evolution_table.tex')

    elif PLOT_TYPE == 'base100':
        print(f"\n--- Chart: NORMALIZED Evolution (Base 100) of Top {NUM_CHALLENGES_TO_PLOT} Challenges ---")
        
        # Each series has the first value as 100
        plot_crosstab = temporal_crosstab.copy().astype(float)
        plot_crosstab = plot_crosstab.div(plot_crosstab.iloc[0]) * 100
        
        y_axis_label = 'Index (Base = 100 in First Year)'
        plot_title = f'Normalized Evolution (Base 100) of Top {NUM_CHALLENGES_TO_PLOT} Challenges'
        plot_crosstab.to_latex(f'{config.RESULTS_FOLDER}/q4_14_1_challenges_base100_evolution_table.tex')

    else:  # 'absolute'
        print(f"\n--- Chart: ABSOLUTE Evolution of Top {NUM_CHALLENGES_TO_PLOT} Challenges ---")
        plot_crosstab = temporal_crosstab
        y_axis_label = 'Number of Mentions'
        plot_title = f'Evolution of Challenges Over the Last 10 Years'
        plot_crosstab.to_latex(f'{config.RESULTS_FOLDER}/q4_14_1_challenges_absolute_evolution_table.tex')

    # --- 4. PREPARATION FOR PLOTTING ---
    top_to_plot = sorted_columns[:NUM_CHALLENGES_TO_PLOT]
    
    df_plot = plot_crosstab[top_to_plot].reset_index().melt(
        id_vars=year_col, 
        var_name='Challenge Category', 
        value_name=y_axis_label
    )
    
    # --- 5. PLOTTING ---
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(24, 16))
    
    sns.lineplot(
        data=df_plot,
        x=year_col,
        y=y_axis_label,
        hue='Challenge Category',
        style='Challenge Category',
        markers=True,
        dashes=False,
        linewidth=2.5,
        markersize=16
    )
    
    plt.title(plot_title, fontsize=ft1)
    plt.xlabel('Year', fontsize=ft2)
    plt.ylabel(y_axis_label, fontsize=ft2)

    plt.legend(
        loc='upper center', 
        bbox_to_anchor=(0.5, -0.15),
        ncol=2,
        fontsize=ft3,
        title_fontsize='x-large'
    )

    plt.xticks(rotation=45, fontsize=ft3)
    plt.yticks(fontsize=ft3)

    # --- 6. SPECIFIC AXIS FORMATTING ---
    if PLOT_TYPE == 'relative':
        from matplotlib.ticker import PercentFormatter
        plt.gca().yaxis.set_major_formatter(PercentFormatter())

    # --- 7. SAVE AND SHOW ---
    plt.savefig(f'{config.RESULTS_FOLDER}/q4_14_1_challenges_over_time_chart_{PLOT_TYPE}.pdf', dpi=300, bbox_inches='tight')
    plt.show()



In [ ]:
##########################################################
#### Cell 13.1: same charts as previous cell, but stacked in a single image
##########################################################
from matplotlib.ticker import MultipleLocator
import textwrap

# --- Font size definition ---
ft1 = 40  # Font for main title
ft2 = 38  # Font for axis titles (X and Y)
ft3 = 34  # Font for axis labels/ticks (X and Y)

# --- Name of the column with challenges already grouped by script 6 ---
grouped_challenge_col = f'ext_{config.LLM_ADVANCED_MODEL_SAFE_NAME}_challenge_group'

# --- General configurations ---
NUM_CHALLENGES_TO_PLOT = 20  # How many challenges to plot
challenge_col = f'ext_{config.LLM_ADVANCED_MODEL_SAFE_NAME}_challenge_group'
year_col = 'year'
doi_original_col = 'doi_original'

# --- Verification ---
if challenge_col not in df_analysis.columns or year_col not in df_analysis.columns:
    print("ERROR: Expected columns not found. Run previous cells.")
else:
    # --- 1. DATA PREPARATION ---
    df_temporal = df_analysis.dropna(subset=[challenge_col, year_col])
    df_temporal[year_col] = df_temporal[year_col].astype(int)
    df_temporal = df_temporal[df_temporal[year_col] >= 2014]


    # Filters null or 'Uncategorized' data in grouped column
    df_ranked = df_analysis.dropna(subset=[grouped_challenge_col])
    df_ranked = df_ranked[df_ranked[grouped_challenge_col] != 'Uncategorized']
    
    # Creates ranking using value_counts() on grouped column
    challenge_ranking = df_ranked[grouped_challenge_col].value_counts().head(20)
    
    top_challenges = challenge_ranking.head(NUM_CHALLENGES_TO_PLOT).index
    df_temporal_top = df_temporal[df_temporal[challenge_col].isin(top_challenges)]

    # --- 2. ABSOLUTE FREQUENCIES ---
    absolute_crosstab = pd.crosstab(
        df_temporal_top[year_col],
        df_temporal_top[challenge_col]
    )

    # --- 3. RELATIVE FREQUENCIES (% per year) ---
    total_pubs_per_year = df_temporal.groupby(year_col)[doi_original_col].nunique()
    challenge_pubs_per_year = df_temporal_top.groupby([year_col, challenge_col])[doi_original_col].nunique()
    numerator_crosstab = challenge_pubs_per_year.unstack(fill_value=0)
    relative_crosstab = numerator_crosstab.div(total_pubs_per_year, axis=0).fillna(0) * 100

    # --- Order columns consistently ---
    sorted_columns = [col for col in top_challenges if col in absolute_crosstab.columns]
    absolute_crosstab = absolute_crosstab[sorted_columns]
    relative_crosstab = relative_crosstab[sorted_columns]

    # --- 4. CONVERSION TO LONG FORMAT (for sns.lineplot) ---
    df_abs = absolute_crosstab.reset_index().melt(
        id_vars=year_col, var_name='Challenge Category', value_name='Number of Mentions'
    )
    df_rel = relative_crosstab.reset_index().melt(
        id_vars=year_col, var_name='Challenge Category', value_name='% of Publications'
    )

    import matplotlib as mpl

    mpl.rcParams['axes.edgecolor'] = 'black'
    mpl.rcParams['xtick.color'] = 'black'
    mpl.rcParams['ytick.color'] = 'black'
    mpl.rcParams['grid.color'] = 'black'
    mpl.rcParams['grid.linewidth'] = 0.2
    mpl.rcParams['patch.edgecolor'] = 'black'


    # --- 5. STYLE CONFIGURATION ---
    #plt.style.use('seaborn-v0_8')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(23, 20), sharex=True, gridspec_kw={'hspace': 0.15})

    # Forces white background on figure and axes
    fig.patch.set_facecolor('white')
    ax1.set_facecolor('white')
    ax2.set_facecolor('white')

    # --- 6. CHART 1: ABSOLUTE ---
    sns.lineplot(
        data=df_abs,
        x=year_col,
        y='Number of Mentions',
        hue='Challenge Category',
        style='Challenge Category',
        markers=True,
        dashes=False,
        linewidth=5,
        markersize=28,
        palette="viridis",  
        ax=ax1
    )

          
    ax1.set_title('Evolution of Challenges (Absolute Values)', fontsize=ft1)
    ax1.set_ylabel('Number of Mentions', fontsize=ft2)
    ax1.set_xlabel('')
    ax1.tick_params(axis='both', labelsize=ft3)
    ax1.yaxis.set_major_locator(MultipleLocator(1000))

    # --- 7. CHART 2: RELATIVE ---
    sns.lineplot(
        data=df_rel,
        x=year_col,
        y='% of Publications',
        hue='Challenge Category',
        style='Challenge Category',
        markers=True,
        dashes=False,
        linewidth=5,
        markersize=28,
        palette="viridis",
        ax=ax2,
        legend=False  # legend will be unified
    )

    ax2.set_title('Share of Publications by Challenge (Relative %)', fontsize=ft1)
    ax2.set_xlabel('Year', fontsize=ft2)
    ax2.set_ylabel('% of Publications', fontsize=ft2)
    ax2.tick_params(axis='both', labelsize=ft3)


    # --- GRID AND BORDERS (apply AFTER lineplots) ---
    ax1.grid(True, axis='y', color='black', linewidth=1)
    ax2.grid(True, axis='y', color='black', linewidth=1)

    for ax in [ax1, ax2]:
        for spine in ax.spines.values():
            spine.set_color('black')
            spine.set_linewidth(2.2)
            
    from matplotlib.ticker import PercentFormatter
    ax2.yaxis.set_major_formatter(PercentFormatter())

    # --- 8. REMOVE LOCAL LEGEND FROM FIRST CHART ---
    ax1.get_legend().remove()  # Removes automatic legend created by Seaborn

    # --- 9. GLOBAL LEGEND BELOW CHARTS ---
    handles, labels = ax1.get_legend_handles_labels()

    labels_2lines = [textwrap.fill(l, width=32) for l in labels]
    fig.legend(
        handles,
        labels_2lines,
        loc='lower center',           # Centers horizontally
        bbox_to_anchor=(0.5, -0.24),  # Increases space between chart and legend
        ncol=2,
        fontsize=ft3,
        title_fontsize='x-large',
        frameon=False 
    )

    # --- 10. FINAL LAYOUT ADJUSTMENT ---
    plt.subplots_adjust(bottom=0.18, hspace=0.15)  # Reserves space for legend and adjusts vertical spacing

    # --- 11. GLOBAL TITLE (optional) ---
    fig.suptitle(
        f'Evolution of Challenges in the Literature',
        fontsize=ft1+15,
        y=0.95
    )


#################################################
# saves as svg to later convert to pdf because pdf is not saving with very fine grid lines
    # --- 12. SAVE AND SHOW ---
    # plt.savefig(
    #     f'{config.RESULTS_FOLDER}/q4_14_1_challenges_absolute_relative_combined.svg',
    #     dpi=600,
    #     bbox_inches='tight'
    # )

    plt.savefig(
        f'{config.RESULTS_FOLDER}/q4_14_1_challenges_absolute_relative_combined.pdf',
        dpi=600,
        bbox_inches='tight'
    )

    plt.show()



#############################################################
###### The latex table is generated in the next cell to avoid cluttering the plotting code

In [ ]:
#######################################################
### generates latex table for the chart in the previous cell
#######################################################

def generate_combined_latex_table_inverted(
    abs_tbl,
    rel_tbl,
    caption,
    label,
    output_path
):
    challenges = abs_tbl.columns.tolist()
    years = abs_tbl.index.tolist()

    n_challenges = len(challenges)

    with open(output_path, "w") as f:
        f.write(r"\begin{landscape}" + "\n")
        f.write(r"\begin{table}[p]" + "\n")
        f.write(r"\centering" + "\n")
        f.write(r"\caption{" + caption + "}" + "\n")
        f.write(r"\label{" + label + "}" + "\n")

        # Column 1: Absolute/Relative (rotated)
        # Column 2: Challenge
        # Rest: years
        col_format = "c l" + " r" * len(years)
        f.write(r"\begin{tabular}{" + col_format + "}\n")
        f.write(r"\toprule" + "\n")

        # Header
        f.write(" & Challenge & " + " & ".join(map(str, years)) + r" \\" + "\n")
        f.write(r"\midrule" + "\n")

        # -------- ABSOLUTE --------
        f.write(
            r"\multirow{" + str(n_challenges) + r"}{*}"
            r"{\rotatebox{90}{\textbf{Absolute}}}"
        )

        for i, ch in enumerate(challenges):
            row = [str(int(abs_tbl.loc[year, ch])) for year in years]

            if i == 0:
                f.write(f" & {ch} & " + " & ".join(row) + r" \\" + "\n")
            else:
                f.write(f" & {ch} & " + " & ".join(row) + r" \\" + "\n")

        # Thick separator line
        f.write(r"\midrule\midrule" + "\n")

        # -------- RELATIVE --------
        f.write(
            r"\multirow{" + str(n_challenges) + r"}{*}"
            r"{\rotatebox{90}{\textbf{Relative}}}"
        )

        for i, ch in enumerate(challenges):
            row = [f"{rel_tbl.loc[year, ch]:.1f}" for year in years]

            if i == 0:
                f.write(f" & {ch} & " + " & ".join(row) + r" \\" + "\n")
            else:
                f.write(f" & {ch} & " + " & ".join(row) + r" \\" + "\n")

        f.write(r"\bottomrule" + "\n")
        f.write(r"\end{tabular}" + "\n")
        f.write(r"\end{table}" + "\n")
        f.write(r"\end{landscape}" + "\n")



generate_combined_latex_table_inverted(
    abs_tbl=absolute_crosstab,
    rel_tbl=relative_crosstab.round(1),
    caption="Evolution of challenges in the literature: absolute counts and relative share by year",
    label="tbl:q4_14_1_challenges_absolute_relative_combined",
    output_path=f"{config.RESULTS_FOLDER}/q4_14_1_challenges_absolute_relative_combined.tex"
)        

### Temporal Trend Analysis of Challenges

In this section, we will deepen the temporal analysis to identify growth, stability, and decline trends of research challenges over the years. For this, we calculate the slope of a linear regression for the time series of each challenge. Results are presented in three distinct charts.

In [ ]:
##########################################################
#### Cell 15
##########################################################
 
# Imports numpy library for regression calculations
import numpy as np

# --- CHANGE HERE: Font size definition ---
ft1 = 30 # Font for main title
ft2 = 24 # Font for axis titles (X and Y) and subtitles
ft3 = 20 # Font for axis labels/ticks (X and Y)

# Column keys
challenge_col = 'challenge_grouped'
year_col = 'year'

if challenge_col not in df_analysis.columns or year_col not in df_analysis.columns:
    print("ERROR: 'challenge_grouped' or 'year' columns not found. Run previous cells.")
else:
    # --- 1. DATA PREPARATION ---
    # ... (data preparation and trend calculation remain the same) ...
    df_temporal = df_analysis.dropna(subset=[challenge_col, year_col]).copy()
    df_temporal[year_col] = df_temporal[year_col].astype(int)
    df_temporal = df_temporal[df_temporal[year_col] >= 2000]

    full_temporal_crosstab = pd.crosstab(
        df_temporal[year_col],
        df_temporal[challenge_col]
    )

    trends = {}
    for challenge in full_temporal_crosstab.columns:
        years = full_temporal_crosstab.index.values
        mentions = full_temporal_crosstab[challenge].values
        slope, intercept = np.polyfit(years, mentions, 1)
        trends[challenge] = {'slope': slope, 'total_mentions': mentions.sum()}

    df_trends = pd.DataFrame.from_dict(trends, orient='index')

    # --- 2. IDENTIFICATION OF TOP 5 IN EACH CATEGORY ---
    growing_challenges = df_trends.sort_values('slope', ascending=False).head(5).index
    declining_challenges = df_trends.sort_values('slope', ascending=True).head(5).index
    top_frequent = df_trends.sort_values('total_mentions', ascending=False).head(30)
    stable_challenges = top_frequent.reindex(top_frequent['slope'].abs().sort_values().index).head(5).index

    print("\n--- Identified Trend Categories ---")
    print(f"Growing Challenges: {list(growing_challenges)}")
    print(f"Stable and Frequent Challenges: {list(stable_challenges)}")
    print(f"Declining Challenges: {list(declining_challenges)}")

    # --- 3. GENERATION OF 3 CHARTS ---
    
    fig, axes = plt.subplots(3, 1, figsize=(18, 28), sharex=True) # Increased height for better visualization
    
    # --- CHANGE HERE: Main title with ft1 font ---
    fig.suptitle('Trend Analysis of Research Challenges Over Time', fontsize=ft1)

    # Chart 1: Growing Challenges
    sns.lineplot(data=full_temporal_crosstab[growing_challenges], dashes=False, linewidth=2.5, markers=True, markersize=8, ax=axes[0])
    axes[0].set_title('Top 5 Challenges with Highest Growth Trend', fontsize=ft2)
    axes[0].set_ylabel('Number of Mentions', fontsize=ft2)
    axes[0].legend(title='Challenge Category', fontsize='large')

    # Chart 2: Stable Challenges
    sns.lineplot(data=full_temporal_crosstab[stable_challenges], dashes=False, linewidth=2.5, markers=True, markersize=8, ax=axes[1])
    axes[1].set_title('Top 5 Frequent & Stable Challenges', fontsize=ft2)
    axes[1].set_ylabel('Number of Mentions', fontsize=ft2)
    axes[1].legend(title='Challenge Category', fontsize='large')

    # Chart 3: Declining Challenges
    sns.lineplot(data=full_temporal_crosstab[declining_challenges], dashes=False, linewidth=2.5, markers=True, markersize=8, ax=axes[2])
    axes[2].set_title('Top 5 Challenges with Declining Trend', fontsize=ft2)
    axes[2].set_ylabel('Number of Mentions', fontsize=ft2)
    axes[2].set_xlabel('Year', fontsize=ft2)
    axes[2].legend(title='Challenge Category', fontsize='large')
    
    # --- CHANGE HERE: Applies ft3 font size for all axis labels ---
    for ax in axes:
        ax.tick_params(axis='y', labelsize=ft3)
        ax.tick_params(axis='x', labelsize=ft3)
    
    plt.xticks(rotation=45)
    plt.tight_layout(rect=[0, 0.03, 1, 0.97]) # Adjusts for titles
    plt.show()

In [ ]:
#######################################
# Hierarchical clustering study of directions
#######################################

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
import numpy as np

import os
import sys
sys.path.append(os.path.abspath("../02_scripts"))

import config

# --- 0. DEFINITIONS AND PLACEHOLDERS ---
# (MODIFIED) - Adjust these values as necessary
TOP_N_COUNTRIES = 20 
TOP_N_DIRECTIONS = 50 # Previously TOP_N_CHALLENGES


# Font size definitions (copied from original)
fs1 = 70
fs2 = 45
fs3 = 30
fs4 = 45

# (MODIFIED) - Keys for new columns
direction_col = 'directions_group' # New: was challenge_col
country_col = 'first_country'      # Same as before

# --- 1. (NEW) LOADING AND JOINING DATA ---
print("--- Loading Parquet files ---")
df_data = pd.read_parquet(f'{config.DATA_FOLDER}/df_data.parquet')
df_phrases = pd.read_parquet(f'{config.DATA_FOLDER}/df_data_phrases.parquet')

print("Files loaded successfully.")

# Joining dataframes to recreate analysis structure
# We want a DataFrame where each row has the country and the 'direction'
print("--- Joining DataFrames (Merge) ---")
df_analysis_new = pd.merge(
    df_data[['doi', country_col]],
    df_phrases[['doi_original', direction_col]],
    left_on='doi',
    right_on='doi_original'
)
print(f"Total pairs (Country, Direction) found: {len(df_analysis_new)}")

# --- 2. DATA PREPARATION AND SORTING (Adapted from original) ---
print("--- Filtering and preparing data ---")
df_filtered = df_analysis_new.dropna(subset=[direction_col, country_col])
df_filtered = df_filtered[df_filtered[country_col] != "UNKNOWN"]

# # 1. Get unique (country, doi) pairs from filtered dataset
# print("--- Calculating Top Countries by unique DOI count ---")
# df_unique_dois_per_country = df_filtered[['doi', country_col]].drop_duplicates()
# # 2. Count DOIs by country and get Top N (THIS IS DESCENDING ORDER OF DOIs)
# country_doi_counts = df_unique_dois_per_country[country_col].value_counts()
# top_countries_sorted = country_doi_counts.head(TOP_N_COUNTRIES).index

# print(f"Top {TOP_N_COUNTRIES} Countries (by DOI count): \n", country_doi_counts.head(TOP_N_COUNTRIES))

# # (Original) Directions calculation remains the same (based on pairs)
# top_directions_sorted = df_filtered[direction_col].value_counts().head(TOP_N_DIRECTIONS).index

# (MODIFIED) - Uses new columns and TOP_N_DIRECTIONS
top_countries_sorted = df_filtered[country_col].value_counts().head(TOP_N_COUNTRIES).index
top_directions_sorted = df_filtered[direction_col].value_counts().head(TOP_N_DIRECTIONS).index

df_heatmap = df_filtered[
    (df_filtered[country_col].isin(top_countries_sorted)) &
    (df_filtered[direction_col].isin(top_directions_sorted))
]

# --- 3. CONTINGENCY TABLE CREATION (Adapted from original) ---
print("--- Creating Contingency Table (Crosstab) ---")
# (MODIFIED) - Uses 'direction_col'
country_direction_crosstab = pd.crosstab(
    df_heatmap[country_col],
    df_heatmap[direction_col]
)
# Reorders rows and columns for consistency
country_direction_crosstab = country_direction_crosstab.reindex(
    index=top_countries_sorted.intersection(country_direction_crosstab.index),
    columns=top_directions_sorted.intersection(country_direction_crosstab.columns)
)

print("\n--- Count Table (Directions) ---")
print(country_direction_crosstab.head())

# (MODIFIED) - Applies the same removal logic to new crosstab
uncategorized_col_name = 'Uncategorized' 
if uncategorized_col_name in country_direction_crosstab.columns:
    country_direction_crosstab = country_direction_crosstab.drop(columns=[uncategorized_col_name])
    print(f"\n--- Column '{uncategorized_col_name}' removed from analysis ---")

##################################################################
#### START OF CLUSTERING CODE (Almost identical to original)
##################################################################

print("\n--- Starting Hierarchical Clustering ---")

# --- STEP 4: PROFILE NORMALIZATION (BY ROW) ---
# (MODIFIED) - Uses new 'country_direction_crosstab'
print("--- Normalizing profile by row (country) ---")
row_totals = country_direction_crosstab.sum(axis=1)

countries_with_data = row_totals[row_totals > 0].index
country_direction_crosstab_filtered = country_direction_crosstab.loc[countries_with_data]
row_totals_filtered = row_totals.loc[countries_with_data]

df_profile = country_direction_crosstab_filtered.div(row_totals_filtered, axis=0)
df_profile = df_profile.fillna(0)

print("\n--- Profile Table (Row Normalized) ---")
print(df_profile.head())

# --- STEP 5: GENERATE CLUSTERMAP ---
print("\n--- Generating Clustermap (Heatmap + Dendrogram) ---")

cbar_position_horizontal = (0.80, 0.95, 0.5, 0.03)

# 'df_profile' is the input, so cluster code works
g = sns.clustermap(
    df_profile,
    method='ward',
    metric='euclidean',
    cmap='YlGnBu', 
    figsize=(20, 22),
    annot=False,
    cbar_pos=cbar_position_horizontal,
    cbar_kws={'orientation': 'horizontal'}
)

for dendro_ax in [g.ax_row_dendrogram, g.ax_col_dendrogram]:
    for line in dendro_ax.collections:
        line.set_linewidth(4)

g.cax.set_xlabel('Proportion of the Country’s\nResearch Effort', fontsize=fs2)
g.cax.tick_params(labelsize=fs3)

# (MODIFIED) - Updates titles and labels for "Direction"
g.figure.suptitle('Hierarchical Clustering of Countries by\n Research Directions Profile', y=1.08, fontsize=fs1)
g.ax_heatmap.set_xlabel('Research Directions Category', fontsize=fs2) # Was 'Challenge'
g.ax_heatmap.set_ylabel('Country', fontsize=fs2)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=fs4)
plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=fs4)

# (MODIFIED) - Updates filename
cluster_filename = f'{config.RESULTS_FOLDER}/q4_14_2_hierarchical_clustermap_profiles_directions.pdf'
plt.savefig(cluster_filename, dpi=300, bbox_inches='tight')
print(f"Clustermap saved to: {cluster_filename}")
plt.show()

# --- STEP 6: (Alternative) Plot Only Dendrogram ---
print("\n--- Generating only Dendrogram (Alternative) ---")

Z = linkage(df_profile, method='ward', metric='euclidean')

plt.figure(figsize=(12, 25))
dendrogram(
    Z,
    labels=df_profile.index,
    orientation='right',
    leaf_font_size=14,
)

# (MODIFIED) - Updates title
plt.title('Similarity Dendrogram of Research Profile by Country (Directions)', fontsize=18, pad=20)
plt.xlabel('Euclidean Distance ("Ward" Clustering)', fontsize=14)
plt.ylabel('Country', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.6)

# (MODIFIED) - Updates filename
dendro_filename = f'{config.RESULTS_FOLDER}/q4_14_2_hierarchical_dendrogram_only_directions.pdf'
plt.savefig(dendro_filename, dpi=300, bbox_inches='tight')
print(f"Dendrogram saved to: {dendro_filename}")
plt.show()

print("\n--- Process Completed ---")

In [ ]:
##################################################################
# Hierarchical clustering study of controversies by country
##################################################################

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
import numpy as np

import os
import sys
sys.path.append(os.path.abspath("../02_scripts"))

import config

# --- 0. DEFINITIONS AND PLACEHOLDERS ---
# (MODIFIED) - Adjust these values as necessary
TOP_N_COUNTRIES = 20 
TOP_N_CONTROVERSIES = 50 # Previously TOP_N_CHALLENGES


# Font size definitions (copied from original)
fs1 = 70
fs2 = 45
fs3 = 30
fs4 = 45

# (MODIFIED) - Keys for new columns
direction_col = 'controversies_group' # New: was challenge_col
country_col = 'first_country'      # Same as before

# --- 1. (NEW) LOADING AND JOINING DATA ---
print("--- Loading Parquet files ---")
df_data = pd.read_parquet(f'{config.DATA_FOLDER}/df_data.parquet')
df_phrases = pd.read_parquet(f'{config.DATA_FOLDER}/df_data_phrases.parquet')

print("Files loaded successfully.")

# Joining dataframes to recreate analysis structure
# We want a DataFrame where each row has the country and the 'direction'
print("--- Joining DataFrames (Merge) ---")
df_analysis_new = pd.merge(
    df_data[['doi', country_col]],
    df_phrases[['doi_original', direction_col]],
    left_on='doi',
    right_on='doi_original'
)
print(f"Total pairs (Country, Direction) found: {len(df_analysis_new)}")

# --- 2. DATA PREPARATION AND SORTING (Adapted from original) ---
print("--- Filtering and preparing data ---")
df_filtered = df_analysis_new.dropna(subset=[direction_col, country_col])
df_filtered = df_filtered[df_filtered[country_col] != "UNKNOWN"]

# # 1. Get unique (country, doi) pairs from filtered dataset
# print("--- Calculating Top Countries by unique DOI count ---")
# df_unique_dois_per_country = df_filtered[['doi', country_col]].drop_duplicates()
# # 2. Count DOIs by country and get Top N (THIS IS DESCENDING ORDER OF DOIs)
# country_doi_counts = df_unique_dois_per_country[country_col].value_counts()
# top_countries_sorted = country_doi_counts.head(TOP_N_COUNTRIES).index

# print(f"Top {TOP_N_COUNTRIES} Countries (by DOI count): \n", country_doi_counts.head(TOP_N_COUNTRIES))

# # (Original) Controversies calculation remains the same (based on pairs)
# top_controversies_sorted = df_filtered[direction_col].value_counts().head(TOP_N_CONTROVERSIES).index

# (MODIFIED) - Uses new columns and TOP_N_CONTROVERSIES
top_countries_sorted = df_filtered[country_col].value_counts().head(TOP_N_COUNTRIES).index
top_controversies_sorted = df_filtered[direction_col].value_counts().head(TOP_N_CONTROVERSIES).index

df_heatmap = df_filtered[
    (df_filtered[country_col].isin(top_countries_sorted)) &
    (df_filtered[direction_col].isin(top_controversies_sorted))
]

# --- 3. CONTINGENCY TABLE CREATION (Adapted from original) ---
print("--- Creating Contingency Table (Crosstab) ---")
# (MODIFIED) - Uses 'direction_col'
country_direction_crosstab = pd.crosstab(
    df_heatmap[country_col],
    df_heatmap[direction_col]
)
# Reorders rows and columns for consistency
country_direction_crosstab = country_direction_crosstab.reindex(
    index=top_countries_sorted.intersection(country_direction_crosstab.index),
    columns=top_controversies_sorted.intersection(country_direction_crosstab.columns)
)

print("\n--- Count Table (Controversies) ---")
print(country_direction_crosstab.head())

# (MODIFIED) - Applies the same removal logic to new crosstab
uncategorized_col_name = 'Uncategorized' 
if uncategorized_col_name in country_direction_crosstab.columns:
    country_direction_crosstab = country_direction_crosstab.drop(columns=[uncategorized_col_name])
    print(f"\n--- Column '{uncategorized_col_name}' removed from analysis ---")

##################################################################
#### START OF CLUSTERING CODE (Almost identical to original)
##################################################################

print("\n--- Starting Hierarchical Clustering ---")

# --- STEP 4: PROFILE NORMALIZATION (BY ROW) ---
# (MODIFIED) - Uses new 'country_direction_crosstab'
print("--- Normalizing profile by row (country) ---")
row_totals = country_direction_crosstab.sum(axis=1)

countries_with_data = row_totals[row_totals > 0].index
country_direction_crosstab_filtered = country_direction_crosstab.loc[countries_with_data]
row_totals_filtered = row_totals.loc[countries_with_data]

df_profile = country_direction_crosstab_filtered.div(row_totals_filtered, axis=0)
df_profile = df_profile.fillna(0)

print("\n--- Profile Table (Row Normalized) ---")
print(df_profile.head())

# --- STEP 5: GENERATE CLUSTERMAP ---
print("\n--- Generating Clustermap (Heatmap + Dendrogram) ---")

cbar_position_horizontal = (0.80, 0.95, 0.5, 0.03)

# adjusts formatting of controversies names
df_profile.columns = df_profile.columns.str.replace("MoS2", r"MoS$_2$", regex=False)
df_profile.columns = df_profile.columns.str.replace("Her ", "HER ", regex=False)


# 'df_profile' is the input, so cluster code works
g = sns.clustermap(
    df_profile,
    method='ward',
    metric='euclidean',
    cmap='YlGnBu', 
    figsize=(20, 22),
    annot=False,
    cbar_pos=cbar_position_horizontal,
    cbar_kws={'orientation': 'horizontal'}
)
g.cax.set_xlabel('Proportion of the Country’s\nResearch Effort', fontsize=fs2)
g.cax.tick_params(labelsize=fs3)

# (MODIFIED) - Updates titles and labels for "Direction"
g.figure.suptitle('Hierarchical Clustering of Countries by\n Research Controversies Profile', y=1.08, fontsize=fs1)
g.ax_heatmap.set_xlabel('Research Controversy Category', fontsize=fs2) # Was 'Challenge'
g.ax_heatmap.set_ylabel('Country', fontsize=fs2)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=fs4)
plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=fs4)

# (MODIFIED) - Updates filename
cluster_filename = f'{config.RESULTS_FOLDER}/q4_14_3_hierarchical_clustermap_profiles_controversies.pdf'
plt.savefig(cluster_filename, dpi=300, bbox_inches='tight')
print(f"Clustermap saved to: {cluster_filename}")
plt.show()

# --- STEP 6: (Alternative) Plot Only Dendrogram ---
print("\n--- Generating only Dendrogram (Alternative) ---")

Z = linkage(df_profile, method='ward', metric='euclidean')

plt.figure(figsize=(12, 25))
dendrogram(
    Z,
    labels=df_profile.index,
    orientation='right',
    leaf_font_size=14,
)

# (MODIFIED) - Updates title
plt.title('Similarity Dendrogram of Research Profile by Country (Controversies)', fontsize=18, pad=20)
plt.xlabel('Euclidean Distance ("Ward" Clustering)', fontsize=14)
plt.ylabel('Country', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.6)

# (MODIFIED) - Updates filename
dendro_filename = f'{config.RESULTS_FOLDER}/q4_14_3_hierarchical_dendrogram_only_controversies.pdf'
plt.savefig(dendro_filename, dpi=300, bbox_inches='tight')
print(f"Dendrogram saved to: {dendro_filename}")
plt.show()

print("\n--- Process Completed ---")

In [ ]:
##################################################################
# Hierarchical clustering study of challenges by topic
##################################################################

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.cluster.hierarchy import linkage, dendrogram
import numpy as np

import os
import sys
sys.path.append(os.path.abspath("../02_scripts"))

import config

import textwrap

def wrap_labels(labels, width=20):
    return ['\n'.join(textwrap.wrap(str(label), width)) for label in labels]


# --- 0. DEFINITIONS AND PLACEHOLDERS ---
# (MODIFIED) - Adjust these values as necessary
TOP_N_TOPICS = 50 
TOP_N_CHALLENGES = 50 # Previously TOP_N_CHALLENGES

# Font size definitions (copied from original)
fs1 = 80
fs2 = 60
fs3 = 60
fs4 = 60

# (MODIFIED) - Keys for new columns
direction_col = 'challenge_group' 
topic_col = 'topic_name_cluster3_reduced'      # Same as before

# --- 1. (NEW) LOADING AND JOINING DATA ---
print("--- Loading Parquet files ---")
df_data = pd.read_parquet(f'{config.DATA_FOLDER}/df_data.parquet')
df_phrases = pd.read_parquet(f'{config.DATA_FOLDER}/df_data_phrases.parquet')

print("Files loaded successfully.")

# Joining dataframes to recreate analysis structure
# We want a DataFrame where each row has the country and the 'direction'
print("--- Joining DataFrames (Merge) ---")
df_analysis_new = pd.merge(
    df_data[['doi', topic_col]],
    df_phrases[['doi_original', direction_col]],
    left_on='doi',
    right_on='doi_original'
)
print(f"Total pairs (Country, Direction) found: {len(df_analysis_new)}")

# --- 2. DATA PREPARATION AND SORTING (Adapted from original) ---
print("--- Filtering and preparing data ---")
df_filtered = df_analysis_new.dropna(subset=[direction_col, topic_col])
df_filtered = df_filtered[df_filtered[topic_col] != "UNKNOWN"]

# # 1. Get unique (country, doi) pairs from filtered dataset

# (MODIFIED) - Uses new columns and TOP_N_CHALLENGES
top_topics_sorted = df_filtered[topic_col].value_counts().head(TOP_N_TOPICS).index
top_challenge_sorted = df_filtered[direction_col].value_counts().head(TOP_N_CHALLENGES).index

df_heatmap = df_filtered[
    (df_filtered[topic_col].isin(top_topics_sorted)) &
    (df_filtered[direction_col].isin(top_challenge_sorted))
]

# --- 3. CONTINGENCY TABLE CREATION (Adapted from original) ---
print("--- Creating Contingency Table (Crosstab) ---")
# (MODIFIED) - Uses 'direction_col'
topic_direction_crosstab = pd.crosstab(
    df_heatmap[topic_col],
    df_heatmap[direction_col]
)
# Reorders rows and columns for consistency
topic_direction_crosstab = topic_direction_crosstab.reindex(
    index=top_topics_sorted.intersection(topic_direction_crosstab.index),
    columns=top_challenge_sorted.intersection(topic_direction_crosstab.columns)
)

print("\n--- Count Table (Challenges) ---")
print(topic_direction_crosstab.head())

# (MODIFIED) - Applies the same removal logic to new crosstab
uncategorized_col_name = 'Uncategorized' 
if uncategorized_col_name in topic_direction_crosstab.columns:
    topic_direction_crosstab = topic_direction_crosstab.drop(columns=[uncategorized_col_name])
    print(f"\n--- Column '{uncategorized_col_name}' removed from analysis ---")

##################################################################
#### START OF CLUSTERING CODE (Almost identical to original)
##################################################################

print("\n--- Starting Hierarchical Clustering ---")

# --- STEP 4: PROFILE NORMALIZATION (BY ROW) ---
# (MODIFIED) - Uses new 'topic_direction_crosstab'
print("--- Normalizing profile by row (country) ---")
row_totals = topic_direction_crosstab.sum(axis=1)

topics_with_data = row_totals[row_totals > 0].index
topic_direction_crosstab_filtered = topic_direction_crosstab.loc[topics_with_data]
row_totals_filtered = row_totals.loc[topics_with_data]

df_profile = topic_direction_crosstab_filtered.div(row_totals_filtered, axis=0)
df_profile = df_profile.fillna(0)

print("\n--- Profile Table (Row Normalized) ---")
print(df_profile.head())

# --- STEP 5: GENERATE CLUSTERMAP ---
print("\n--- Generating Clustermap (Heatmap + Dendrogram) ---")

cbar_position_horizontal = (0.80, 0.90, 0.5, 0.03)

# adjusts formatting of challenge names
df_profile.columns = df_profile.columns.str.replace("MoS2", r"MoS$_2$", regex=False)
df_profile.columns = df_profile.columns.str.replace("Her ", "HER ", regex=False)


# 'df_profile' is the input, so cluster code works
g = sns.clustermap(
    df_profile,
    method='ward',
    metric='euclidean',
    cmap='YlGnBu', 
    figsize=(24, 54),
    annot=False,
    cbar_pos=cbar_position_horizontal,
    cbar_kws={'orientation': 'horizontal'}
)

for dendro_ax in [g.ax_row_dendrogram, g.ax_col_dendrogram]:
    for line in dendro_ax.collections:
        line.set_linewidth(4)


g.cax.set_xlabel('Proportion of the Topic’s\nResearch Effort', fontsize=fs2)
g.cax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
g.cax.tick_params(labelsize=fs3)

# (MODIFIED) - Updates titles and labels for "Direction"
g.figure.suptitle('Hierarchical Clustering of Topics by\n Research Challenges Profile', y=1.05, fontsize=fs1)
g.ax_heatmap.set_xlabel('Research Challenge Category', fontsize=fs2) # Was 'Challenge'
g.ax_heatmap.set_ylabel('Topic', fontsize=fs2)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=fs4)


#plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=fs4)
# Gets current labels
yticks = g.ax_heatmap.get_yticklabels()
# Extracts text
ylabels = [tick.get_text() for tick in yticks]
# Applies wrap
wrapped_ylabels = wrap_labels(ylabels, width=35)
# Sets again on axis
g.ax_heatmap.set_yticklabels(wrapped_ylabels, fontsize=fs4)





# (MODIFIED) - Updates filename
cluster_filename = f'{config.RESULTS_FOLDER}/q4_14_5_hierarchical_clustermap_profiles_challenges_by_topics.pdf'
plt.savefig(cluster_filename, dpi=300, bbox_inches='tight')
print(f"Clustermap saved to: {cluster_filename}")
plt.show()

# --- STEP 6: (Alternative) Plot Only Dendrogram ---
print("\n--- Generating only Dendrogram (Alternative) ---")

Z = linkage(df_profile, method='ward', metric='euclidean')

plt.figure(figsize=(12, 25))
dendrogram(
    Z,
    labels=df_profile.index,
    orientation='right',
    leaf_font_size=14,
)

# (MODIFIED) - Updates title
plt.title('Similarity Dendrogram of Research Profile by Topic (Challenges)', fontsize=18, pad=20)
plt.xlabel('Euclidean Distance ("Ward" Clustering)', fontsize=14)
plt.ylabel('Country', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.6)

# (MODIFIED) - Updates filename
dendro_filename = f'{config.RESULTS_FOLDER}/q4_14_5_hierarchical_dendrogram_only_challenges_by_topic.pdf'
plt.savefig(dendro_filename, dpi=300, bbox_inches='tight')
print(f"Dendrogram saved to: {dendro_filename}")
plt.show()

print("\n--- Process Completed ---")

In [ ]:
##################################################################
# Hierarchical clustering study of directions by topic
##################################################################

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.cluster.hierarchy import linkage, dendrogram
import numpy as np

import os
import sys
sys.path.append(os.path.abspath("../02_scripts"))

import config

# --- 0. DEFINITIONS AND PLACEHOLDERS ---
# (MODIFIED) - Adjust these values as necessary
TOP_N_TOPICS = 50 
TOP_N_DIRECTIONS = 50 

# Font size definitions (copied from original)
fs1 = 70
fs2 = 45
fs3 = 30
fs4 = 55

# (MODIFIED) - Keys for new columns
direction_col = 'directions_group' 
topic_col = 'topic_name_cluster3_reduced'      # Same as before

# --- 1. (NEW) LOADING AND JOINING DATA ---
print("--- Loading Parquet files ---")
df_data = pd.read_parquet(f'{config.DATA_FOLDER}/df_data.parquet')
df_phrases = pd.read_parquet(f'{config.DATA_FOLDER}/df_data_phrases.parquet')

print("Files loaded successfully.")

# Joining dataframes to recreate analysis structure
# We want a DataFrame where each row has the country and the 'direction'
print("--- Joining DataFrames (Merge) ---")
df_analysis_new = pd.merge(
    df_data[['doi', topic_col]],
    df_phrases[['doi_original', direction_col]],
    left_on='doi',
    right_on='doi_original'
)
print(f"Total pairs (Country, Direction) found: {len(df_analysis_new)}")

# --- 2. DATA PREPARATION AND SORTING (Adapted from original) ---
print("--- Filtering and preparing data ---")
df_filtered = df_analysis_new.dropna(subset=[direction_col, topic_col])
df_filtered = df_filtered[df_filtered[topic_col] != "UNKNOWN"]

# # 1. Get unique (country, doi) pairs from filtered dataset

# (MODIFIED) - Uses new columns and TOP_N_DIRECTIONS
top_topics_sorted = df_filtered[topic_col].value_counts().head(TOP_N_TOPICS).index
top_directions_sorted = df_filtered[direction_col].value_counts().head(TOP_N_DIRECTIONS).index

df_heatmap = df_filtered[
    (df_filtered[topic_col].isin(top_topics_sorted)) &
    (df_filtered[direction_col].isin(top_directions_sorted))
]

# --- 3. CONTINGENCY TABLE CREATION (Adapted from original) ---
print("--- Creating Contingency Table (Crosstab) ---")
# (MODIFIED) - Uses 'direction_col'
topic_direction_crosstab = pd.crosstab(
    df_heatmap[topic_col],
    df_heatmap[direction_col]
)
# Reorders rows and columns for consistency
topic_direction_crosstab = topic_direction_crosstab.reindex(
    index=top_topics_sorted.intersection(topic_direction_crosstab.index),
    columns=top_directions_sorted.intersection(topic_direction_crosstab.columns)
)

print("\n--- Count Table (Directions) ---")
print(topic_direction_crosstab.head())

# (MODIFIED) - Applies the same removal logic to new crosstab
uncategorized_col_name = 'Uncategorized' 
if uncategorized_col_name in topic_direction_crosstab.columns:
    topic_direction_crosstab = topic_direction_crosstab.drop(columns=[uncategorized_col_name])
    print(f"\n--- Column '{uncategorized_col_name}' removed from analysis ---")

##################################################################
#### START OF CLUSTERING CODE (Almost identical to original)
##################################################################

print("\n--- Starting Hierarchical Clustering ---")

# --- STEP 4: PROFILE NORMALIZATION (BY ROW) ---
# (MODIFIED) - Uses new 'topic_direction_crosstab'
print("--- Normalizing profile by row (country) ---")
row_totals = topic_direction_crosstab.sum(axis=1)

topics_with_data = row_totals[row_totals > 0].index
topic_direction_crosstab_filtered = topic_direction_crosstab.loc[topics_with_data]
row_totals_filtered = row_totals.loc[topics_with_data]

df_profile = topic_direction_crosstab_filtered.div(row_totals_filtered, axis=0)
df_profile = df_profile.fillna(0)

print("\n--- Profile Table (Row Normalized) ---")
print(df_profile.head())

# --- STEP 5: GENERATE CLUSTERMAP ---
print("\n--- Generating Clustermap (Heatmap + Dendrogram) ---")

cbar_position_horizontal = (0.80, 0.95, 0.5, 0.03)

# adjusts formatting of directions names
df_profile.columns = df_profile.columns.str.replace("MoS2", r"MoS$_2$", regex=False)
df_profile.columns = df_profile.columns.str.replace("Her ", "HER ", regex=False)


# 'df_profile' is the input, so cluster code works
g = sns.clustermap(
    df_profile,
    method='ward',
    metric='euclidean',
    cmap='YlGnBu', 
    figsize=(20, 32),
    annot=False,
    cbar_pos=cbar_position_horizontal,
    cbar_kws={'orientation': 'horizontal'}
)

for dendro_ax in [g.ax_row_dendrogram, g.ax_col_dendrogram]:
    for line in dendro_ax.collections:
        line.set_linewidth(4)


g.cax.set_xlabel('Proportion of the Topic’s\nResearch Effort', fontsize=fs2)
g.cax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
g.cax.tick_params(labelsize=fs3)

# (MODIFIED) - Updates titles and labels for "Direction"
g.figure.suptitle('Hierarchical Clustering of Topics by\n Research Directions Profile', y=1.08, fontsize=fs1)
g.ax_heatmap.set_xlabel('Research Directions Category', fontsize=fs2) 
g.ax_heatmap.set_ylabel('Topic', fontsize=fs2)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=fs4)
plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=fs4)

# (MODIFIED) - Updates filename
cluster_filename = f'{config.RESULTS_FOLDER}/q4_14_6_hierarchical_clustermap_profiles_directions_by_topics.pdf'
plt.savefig(cluster_filename, dpi=300, bbox_inches='tight')
print(f"Clustermap saved to: {cluster_filename}")
plt.show()

# --- STEP 6: (Alternative) Plot Only Dendrogram ---
print("\n--- Generating only Dendrogram (Alternative) ---")

Z = linkage(df_profile, method='ward', metric='euclidean')

plt.figure(figsize=(12, 25))
dendrogram(
    Z,
    labels=df_profile.index,
    orientation='right',
    leaf_font_size=14,
)

# (MODIFIED) - Updates title
plt.title('Similarity Dendrogram of Research Profile by Topic (Directions)', fontsize=18, pad=20)
plt.xlabel('Euclidean Distance ("Ward" Clustering)', fontsize=14)
plt.ylabel('Country', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.6)

# (MODIFIED) - Updates filename
dendro_filename = f'{config.RESULTS_FOLDER}/q4_14_6_hierarchical_dendrogram_only_directions_by_topic.pdf'
plt.savefig(dendro_filename, dpi=300, bbox_inches='tight')
print(f"Dendrogram saved to: {dendro_filename}")
plt.show()

print("\n--- Process Completed ---")

---
## Question: Are there unresolved debates about specific catalysts, methods, or efficiency claims?

There is no single chart that answers this question, but we can use a combination of quantitative analysis (looking for variability in the data we already have) and qualitative analysis (using the LLM to find and summarize texts that discuss controversies).

We will explore three complementary approaches. The first one you can implement immediately with the data we already have. The other two are suggestions to deepen the analysis.

#### Approach 1: Analysis of Inconsistency in Performance Data (Immediate Implementation)
Idea: If there is a "debate over efficiency claims", this should manifest as a large variation in the performance values reported for the same material across different articles. A well-understood catalyst will have consistent performance values in the literature, while a controversial one will have a large dispersion.

Suggested Chart: Box Plot of Performance by Catalyst
A box plot is the perfect tool to visualize the distribution (median, quartiles, outliers) of a variable. We will create a box plot of overpotential_mv for the 10 most cited catalysts.

How to interpret:

Wide and long boxes: Indicate great variability and inconsistency in the results reported for that material. This is a strong sign of debate!

Short and compact boxes: Indicate a consensus in the literature about the performance of that material.

Many points outside the box (outliers): Can indicate exceptional (or very poor) efficiency claims that fall outside the consensus, another sign of debate.

In [ ]:
#################################################################
####  Cell: Efficiency Inconsistency Box Plot

# --- CHANGE HERE: Imports regular expressions library ---
import re

# --- CHANGE HERE: Creates a function to format chemical formulas ---
def format_chemical_formula(name):
    """
    Uses regex to find numbers in a string and format them as subscripts
    for matplotlib mathtext. Ex: 'MoS2' -> 'MoS$_2$'
    """
    if not isinstance(name, str):
        return name
    # Finds all digits in the string and wraps them with '$_{...}$'
    return re.sub(r'(\d+)', r'$_{\1}$', name)

# --- The rest of the cell code ---

# Required columns for this analysis
challenge_col = 'challenge_grouped'
material_col = f'ext_{config.LLM_ADVANCED_MODEL_SAFE_NAME}_catalyst_material_main'
overpotential_col = f'ext_{config.LLM_ADVANCED_MODEL_SAFE_NAME}_overpotential_mv'

if material_col not in df_analysis.columns or overpotential_col not in df_analysis.columns:
    print(f"ERROR: Material or overpotential columns not found.")
else:
    print("--- Inconsistency Analysis: Overpotential Distribution by Catalyst ---\n")
    
    # 1. Prepare data (logic remains the same as before)
    df_perf = df_analysis.dropna(subset=[material_col, overpotential_col]).copy()
    df_perf[overpotential_col] = pd.to_numeric(df_perf[overpotential_col], errors='coerce')
    df_perf.dropna(subset=[overpotential_col], inplace=True)
    df_perf = df_perf[df_perf[material_col].str.lower() != 'null']
    
    df_perf['material_normalized'] = df_perf[material_col].str.lower().str.strip()
    top_10_normalized = df_perf['material_normalized'].value_counts().head(10).index
    df_plot_perf = df_perf[df_perf['material_normalized'].isin(top_10_normalized)].copy()
    
    df_plot_perf['group_count'] = df_plot_perf.groupby('material_normalized')['material_normalized'].transform('count')
    df_plot_perf.sort_values('group_count', ascending=False, inplace=True)
    
    # Gets the order of names with original capitalization
    display_order_original = df_plot_perf[material_col].unique()

    # --- CHANGE HERE: Applies formatting for chart axes ---
    # Creates a new column with names formatted for the chart
    df_plot_perf['material_formatted'] = df_plot_perf[material_col].apply(format_chemical_formula)
    
    # Creates a list with the order of formatted names for the boxplot 'order' parameter
    display_order_formatted = [format_chemical_formula(name) for name in display_order_original]

    # 2. CHART GENERATION (Box Plot)
    if not df_plot_perf.empty:
        plt.style.use('seaborn-v0_8-whitegrid')
        plt.figure(figsize=(18, 10))
        
        # --- CHANGE HERE: Uses new column and new order on chart ---
        sns.boxplot(
            data=df_plot_perf,
            x=overpotential_col,
            y='material_formatted',      # Uses column with formatted names
            order=display_order_formatted, # Ensures frequency order with formatted names
            palette='viridis'
        )
        
        plt.title('Distribution of Reported Overpotential for Top 10 Catalysts', fontsize=22)
        plt.xlabel('Overpotential (mV) - Lower is Better', fontsize=16)
        plt.ylabel('Catalyst Material', fontsize=16)
        plt.xticks(fontsize=14)
        plt.yticks(fontsize=14) # The labels here will now be rendered with mathtext
        
        plt.xlim(0, df_plot_perf[overpotential_col].quantile(0.99)) 
        
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough data to generate the performance distribution chart.")